# 1. Training
Hyperparameter search, the proposed model on the five splits, and the same network with other pooling functions.

Run the cells from top to bottom. A run whose result file already exists is skipped, so the notebook can be stopped and started again.

In [ ]:
import torch

from constants import *
from utils import get_device
from experiments import run_experiment
from hparam_search import run_hyperparameter_search
from disk_utils import load_results
from result_tables import per_seed_table, summary_table

In [ ]:
USE_GPU = True
device = get_device(USE_GPU)
print(device)

# QUICK_TEST = True makes a tiny dry run of every step (1 epoch, 3 batches)
# and writes to results_quick_test/. Use it to check the setup.
QUICK_TEST = False

if QUICK_TEST:
    results_dir = QUICK_TEST_RESULTS_DIR
    num_epochs = 1
    max_train_batches = 3
else:
    results_dir = RESULTS_DIR
    num_epochs = NUM_EPOCHS
    max_train_batches = None

if QUICK_TEST:
    num_search_trials = 2
else:
    num_search_trials = NUM_SEARCH_TRIALS

## Hyperparameter search
Random search on split 0. Trial 0 is the configuration that was fixed beforehand. The configurations are compared by validation loss; the test subset is not used.

In [ ]:
search_table = run_hyperparameter_search(
    device,
    num_trials=num_search_trials,
    seed=0,
    num_epochs=num_epochs,
    max_train_batches=max_train_batches,
    results_dir=results_dir
)

In [ ]:
search_table

## Proposed model
BiLSTM with attention on all five note attributes (pitch class, accidental, octave, duration, measure position).

In [ ]:
for seed in SEEDS:
    run_experiment(
        tag="full",
        seed=seed,
        device=device,
        num_epochs=num_epochs,
        max_train_batches=max_train_batches,
        results_dir=results_dir
    )

## Without packing
The LSTM also runs over the padding, as in the preliminary version.

In [ ]:
for seed in SEEDS:
    run_experiment(
        tag="full_unpacked",
        seed=seed,
        device=device,
        packed=False,
        num_epochs=num_epochs,
        max_train_batches=max_train_batches,
        results_dir=results_dir
    )

## Other pooling functions in place of attention

In [ ]:
for seed in SEEDS:
    run_experiment(
        tag="mean_pool",
        seed=seed,
        device=device,
        pooling="mean",
        num_epochs=num_epochs,
        max_train_batches=max_train_batches,
        results_dir=results_dir
    )

In [ ]:
for seed in SEEDS:
    run_experiment(
        tag="max_pool",
        seed=seed,
        device=device,
        pooling="max",
        num_epochs=num_epochs,
        max_train_batches=max_train_batches,
        results_dir=results_dir
    )

In [ ]:
for seed in SEEDS:
    run_experiment(
        tag="last_pool",
        seed=seed,
        device=device,
        pooling="last",
        num_epochs=num_epochs,
        max_train_batches=max_train_batches,
        results_dir=results_dir
    )

## Results so far
Mean accuracy and macro-F1 over the five splits, with the half-width of the 95% confidence interval.

In [ ]:
results = load_results(results_dir)
per_seed = per_seed_table(results)
summary = summary_table(per_seed)
summary[["tag", "perturbation", "n", "acc_mean", "acc_ci", "f1_mean", "f1_ci"]]